# Offline Analysis Demo

Offline/post-processing workflow from an existing run snapshot.


## User Inputs

Edit only the next cell.


In [ ]:
from pathlib import Path

# Directory containing config.json and debate_snapshot.json from a prior run.
source_run_dir = Path("../outputs/promotion_committee_max_divergence_no_incentive")

# Post-processing selections only.
postpro = {
    "semantic_analysis_metrics": [
        "self_consistency",
        "cross_agent_public_alignment",
        "cross_agent_private_alignment",
    ],
    "persona_analysis_metrics": [],
    "save_plots": True,
    "show_plots": False,
    "persona_scoring_model": "anthropic/claude-sonnet-4",
    "persona_scoring_verbose": False,
    "persona_score_samples": 1,
}


## Build Offline Config (Internal)


In [ ]:
import json
from dataclasses import fields

from dotenv import load_dotenv

from agora.experiment import ExperimentConfig, build_experiment_config

load_dotenv()

base_config_path = source_run_dir / "config.json"
snapshot_path = source_run_dir / "debate_snapshot.json"
if not base_config_path.exists():
    raise FileNotFoundError(f"Missing config file: {base_config_path}")
if not snapshot_path.exists():
    raise FileNotFoundError(f"Missing snapshot file: {snapshot_path}")

base_raw = json.loads(base_config_path.read_text(encoding="utf-8"))
allowed = {field.name for field in fields(ExperimentConfig)}
base_config = {k: v for k, v in base_raw.items() if k in allowed}

offline_payload = {
    **base_config,
    "num_turns": 0,
    "load_snapshot": True,
    "load_dir": source_run_dir,
    "save_snapshot": False,
    "reuse_load_dir_for_outputs": True,
    "indexed_output": False,
    **postpro,
}

config = build_experiment_config(offline_payload)
config


## Run Post-Processing


In [ ]:
from agora.experiment import run_persona_experiment

result = run_persona_experiment(config)
print("Output directory:", result.run_dir)
